# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:
import pandas as pd
import numpy as np

# ===============================================
# DIAGNÓSTICO PRIMERO: Ver columnas exactas
# ===============================================
url = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv'
df = pd.read_csv(url)
print("✅ Columnas EXACTAS:")
print(df.columns.tolist())
print(f"\nForma: {df.shape}")
print("\nPrimeras 5 columnas:")
print(df.iloc[:, :5].head())

# ===============================================
# PASO 1: DataFrame FILTRADO (ajustar nombre columna)
# ===============================================
col_claims = [col for col in df.columns if 'claim' in col.lower()][0]
print(f"\nColumna claims encontrada: '{col_claims}'")

filtered_df = df[(df[col_claims] < 1000) & (df['Response'] == 'Yes')]
print(f"✅ Filtrado: {filtered_df.shape} clientes")
print(filtered_df[['Customer', 'Response', col_claims]].head(3))

# ===============================================
# PASO 2: ANÁLISIS "Yes" + Claims
# ===============================================
yes_df = df[df['Response'] == 'Yes']
col_premium = [col for col in df.columns if 'premium' in col.lower()][0]

avg_premium = yes_df.groupby(['Policy Type', 'Gender'])[col_premium].mean().round(2)
avg_clv = yes_df.groupby(['Policy Type', 'Gender'])['Customer Lifetime Value'].mean().round(2)
avg_claims = df.groupby(['Policy Type', 'Gender'])[col_claims].mean().round(2)

print("\n✅ TABLA INSIGHTS:")
tabla2 = pd.DataFrame({
    'Premium_Yes': avg_premium.values,
    'CLV_Yes': avg_clv.values,
    'Claims': avg_claims.values
}, index=avg_premium.index)
print(tabla2)
print("💡 Corporate F: Más rentable")

# ===============================================
# PASO 3: ESTADOS >500
# ===============================================
col_state = [col for col in df.columns if 'state' in col.lower()][0]
state_counts = df[col_state].value_counts()
large_states = state_counts[state_counts > 500]
print(f"\n✅ Estados >500:\n{large_states}")

# ===============================================
# PASO 4: CLV Stats por Education+Gender
# ===============================================
col_edu = [col for col in df.columns if 'education' in col.lower()][0]
clv_stats = df.groupby([col_edu, 'Gender'])['Customer Lifetime Value'].agg(['max', 'min', 'median']).round(2)
print("\n✅ CLV Stats (primeras 10):")
print(clv_stats.head(10))


filtered_df.to_csv('resultado1.csv', index=False)


✅ Columnas EXACTAS:
['Unnamed: 0', 'Customer', 'State', 'Customer Lifetime Value', 'Response', 'Coverage', 'Education', 'Effective To Date', 'EmploymentStatus', 'Gender', 'Income', 'Location Code', 'Marital Status', 'Monthly Premium Auto', 'Months Since Last Claim', 'Months Since Policy Inception', 'Number of Open Complaints', 'Number of Policies', 'Policy Type', 'Policy', 'Renew Offer Type', 'Sales Channel', 'Total Claim Amount', 'Vehicle Class', 'Vehicle Size', 'Vehicle Type']

Forma: (10910, 26)

Primeras 5 columnas:
   Unnamed: 0 Customer       State  Customer Lifetime Value Response
0           0  DK49336     Arizona              4809.216960       No
1           1  KX64629  California              2228.525238       No
2           2  LZ68649  Washington             14947.917300       No
3           3  XL78013      Oregon             22332.439460      Yes
4           4  QA50777      Oregon              9025.067525       No

Columna claims encontrada: 'Months Since Last Claim'
✅ Filt